Imports

In [ ]:
import os
import pandas as pd
import numpy as np

from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms

from sklearn.model_selection import train_test_split

from scipy.stats import pearsonr
from scipy.stats import spearmanr
torch.backends.cudnn.benchmark = True
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


Dataset Paths

In [ ]:
import os
import pandas as pd
import numpy as np

KADID_ROOT = "/content/dataset/kadid10k"

csv_path = os.path.join(KADID_ROOT, "dmos.csv")

df = pd.read_csv(csv_path)

print(df.columns)

records = []

for _, row in df.iterrows():

    dist_img = row["dist_img"]
    ref_img = row["ref_img"]
    dmos = float(row["dmos"])

    records.append({
        "ref_path": os.path.join(
            KADID_ROOT,
            "images",
            ref_img
        ),
        "dist_path": os.path.join(
            KADID_ROOT,
            "images",
            dist_img
        ),
        "mos": dmos
    })

df = pd.DataFrame(records)

print("Total samples:", len(df))

mos = df["mos"].values.astype(np.float32)

mos = (mos - mos.min()) / (
    mos.max() - mos.min() + 1e-8
)

# Convert DMOS -> Quality Score
mos = 1.0 - mos

df["mos"] = mos

print(df.head())

Index(['dist_img', 'ref_img', 'dmos', 'var'], dtype='object')
Total samples: 10125
                                            ref_path  \
0  /content/drive/MyDrive/kadid10k/kadid10k/image...   
1  /content/drive/MyDrive/kadid10k/kadid10k/image...   
2  /content/drive/MyDrive/kadid10k/kadid10k/image...   
3  /content/drive/MyDrive/kadid10k/kadid10k/image...   
4  /content/drive/MyDrive/kadid10k/kadid10k/image...   

                                           dist_path       mos  
0  /content/drive/MyDrive/kadid10k/kadid10k/image...  0.091603  
1  /content/drive/MyDrive/kadid10k/kadid10k/image...  0.152672  
2  /content/drive/MyDrive/kadid10k/kadid10k/image...  0.575064  
3  /content/drive/MyDrive/kadid10k/kadid10k/image...  0.829517  
4  /content/drive/MyDrive/kadid10k/kadid10k/image...  0.974555  


Train / Val / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 7087
Val: 1519
Test: 1519


Dataset

In [ ]:
class KADIDDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

        self.transform = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        ref = Image.open(
            row["ref_path"]
        ).convert("RGB")

        dist = Image.open(
            row["dist_path"]
        ).convert("RGB")

        ref = self.transform(ref)
        dist = self.transform(dist)

        mos = torch.tensor(
            row["mos"],
            dtype=torch.float32
        )

        return ref, dist, mos

Dataloaders

In [ ]:
train_loader = DataLoader(
    KADIDDataset(train_df),
    batch_size=8,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    KADIDDataset(val_df),
    batch_size=8,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    KADIDDataset(test_df),
    batch_size=8,
    shuffle=False,
    num_workers=0
)

select backbone

In [ ]:
BACKBONE = "resnet50_aspp"

# Options:
# "vgg19"
# "vgg19_aspp"
# "resnet50"
# "resnet50_aspp"
# "squeeze"
# "squeeze_aspp"
# "efficientnet"
# "efficientnet_aspp"
# "mobilenetv2"
# "mobilenetv2_aspp"

BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-5

In [ ]:
import os

files = [
    "DBIQA_ResNet.py",
    "DBIQA_VGG.py",
    "DBIQA_Squeeze.py",
    "DBIQA_EfficientNet.py",
    "DBIQA_mobileV2.py"
]

for file_path in files:
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            text = f.read()

        text = text.replace("pretrained=True", "pretrained=False")

        with open(file_path, "w") as f:
            f.write(text)

        print(f"Updated: {file_path}")
    else:
        print(f"Missing: {file_path}")

Updated: DBIQA_ResNet.py
Updated: DBIQA_VGG.py
Updated: DBIQA_Squeeze.py
Updated: DBIQA_EfficientNet.py
Updated: DBIQA_mobileV2.py


 Import Official DBIQA

In [ ]:
if BACKBONE == "vgg19":
    from DBIQA_VGG import DBIQA, VGG

elif BACKBONE == "vgg19_aspp":
    from DBIQA_VGG_ASPP import DBIQA, VGG

elif BACKBONE == "resnet50":
    from DBIQA_ResNet import DBIQA, ResNet50

elif BACKBONE == "resnet50_aspp":
    from DBIQA_ResNet_ASPP import DBIQA, ResNet50

elif BACKBONE == "squeeze":
    from DBIQA_Squeeze import DBIQA, Squeeze

elif BACKBONE == "squeeze_aspp":
    from DBIQA_Squeeze_ASPP import DBIQA, Squeeze

elif BACKBONE == "efficientnet":
    from DBIQA_EfficientNet import DBIQA, efficientB0

elif BACKBONE == "efficientnet_aspp":
    from DBIQA_EfficientNet_ASPP import DBIQA, efficientB0

elif BACKBONE == "mobilenetv2":
    from DBIQA_mobileV2 import DBIQA, mobileV2

elif BACKBONE == "mobilenetv2_aspp":
    from DBIQA_mobileV2_ASPP import DBIQA, mobileV2

else:
    raise ValueError(f"Unknown BACKBONE: {BACKBONE}")

Create Network

In [ ]:
if BACKBONE in ["vgg19", "vgg19_aspp"]:

    feature_net = VGG(
        pretrained=False,
        requires_grad=True
    )

elif BACKBONE in ["resnet50", "resnet50_aspp"]:

    feature_net = ResNet50(
        requires_grad=True
    )

elif BACKBONE in ["squeeze", "squeeze_aspp"]:

    feature_net = Squeeze(
        requires_grad=True
    )

elif BACKBONE in ["mobilenetv2", "mobilenetv2_aspp"]:

    feature_net = mobileV2(
        requires_grad=True
    )

elif BACKBONE in ["efficientnet", "efficientnet_aspp"]:

    feature_net = efficientB0(
        requires_grad=True
    )

else:
    raise ValueError(f"Unknown BACKBONE: {BACKBONE}")

feature_net = feature_net.to(device)
model = DBIQA().to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 169MB/s]


PLCC Loss

In [ ]:
class PLCCLoss(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, pred, target):

        pred = pred.view(-1)
        target = target.view(-1)

        pred = pred - pred.mean()
        target = target - target.mean()

        pred_norm = torch.sqrt(torch.sum(pred**2) + 1e-8)
        target_norm = torch.sqrt(torch.sum(target**2) + 1e-8)

        plcc = torch.sum(pred * target) / (
            pred_norm * target_norm
        )

        return 1 - torch.abs(plcc)



Optimizer

In [ ]:
criterion = PLCCLoss()
optimizer = torch.optim.Adam(
    list(feature_net.parameters()) +
    list(model.parameters()),
    lr=1e-5
)

Validation Function

In [ ]:
def evaluate(loader):

    feature_net.eval()
    model.eval()

    preds = []
    gts = []

    with torch.no_grad():

        for batch_idx, (ref, dist, mos) in enumerate(loader):

          if batch_idx % 5 == 0:
            ref = ref.to(device)
            dist = dist.to(device)

            ref_feats = feature_net(ref)
            dist_feats = feature_net(dist)

            pred = model(
                ref_feats,
                dist_feats,
                as_loss=False
            )

            preds.extend(pred.cpu().numpy())
            gts.extend(mos.numpy())

    preds = np.array(preds)
    gts = np.array(gts)

    plcc = pearsonr(preds, gts)[0]

    srcc = spearmanr(preds, gts)[0]

    return plcc, srcc

Training Loop

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch

SAVE_DIR = "/content/drive/MyDrive/DBIQA_Results/KADID10K"
os.makedirs(SAVE_DIR, exist_ok=True)

history_path = os.path.join(
    SAVE_DIR,
    f"DBIQA_{BACKBONE}_Epochwise_Results.csv"
)

best_checkpoint = os.path.join(
    SAVE_DIR,
    f"best_dbiqa_{BACKBONE}.pth"
)

latest_checkpoint = os.path.join(
    SAVE_DIR,
    f"latest_dbiqa_{BACKBONE}.pth"
)


start_epoch = 0
best_srcc = -1

if os.path.exists(latest_checkpoint):

    checkpoint = torch.load(
        latest_checkpoint,
        map_location=device,
        weights_only=False
    )

    feature_net.load_state_dict(
        checkpoint["feature_net"]
    )

    model.load_state_dict(
        checkpoint["dbiqa"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer"]
    )

    start_epoch = checkpoint["epoch"]
    best_srcc = checkpoint["best_srcc"]

    print(f"\nResuming from Epoch {start_epoch+1}")

else:
    print("\nTraining from Scratch")


if os.path.exists(history_path):
    history = pd.read_csv(history_path).to_dict("records")
else:
    history = []

print(f"\nTraining Backbone : {BACKBONE}")

start_time = time.time()


for epoch in range(start_epoch, EPOCHS):

    feature_net.train()
    model.train()

    running_loss = 0

    for ref, dist, mos in train_loader:

        ref = ref.to(device, non_blocking=True)
        dist = dist.to(device, non_blocking=True)
        mos = mos.to(device, non_blocking=True)

        optimizer.zero_grad()

        ref_feat = feature_net(ref)
        dist_feat = feature_net(dist)

        pred = model(
            ref_feat,
            dist_feat,
            as_loss=True
        ).squeeze()

        loss = criterion(pred, mos)

        if torch.isnan(loss):
            print("NaN Loss!")
            break

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            list(feature_net.parameters()) +
            list(model.parameters()),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    val_plcc, val_srcc = evaluate(val_loader)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Loss={train_loss:.4f} | "
        f"PLCC={val_plcc:.4f} | "
        f"SRCC={val_srcc:.4f}"
    )

    history.append({
        "Epoch": epoch + 1,
        "TrainLoss": train_loss,
        "PLCC": val_plcc,
        "SRCC": val_srcc
    })

    pd.DataFrame(history).to_csv(
        history_path,
        index=False
    )


    latest_state = {

        "epoch": epoch + 1,

        "feature_net": feature_net.state_dict(),

        "dbiqa": model.state_dict(),

        "optimizer": optimizer.state_dict(),

        "best_srcc": best_srcc

    }

    torch.save(
        latest_state,
        latest_checkpoint
    )


    if (not np.isnan(val_srcc)) and (val_srcc > best_srcc):

        best_srcc = val_srcc

        best_state = {

            "epoch": epoch + 1,

            "feature_net": feature_net.state_dict(),

            "dbiqa": model.state_dict(),

            "optimizer": optimizer.state_dict(),

            "best_srcc": best_srcc,

            "val_plcc": val_plcc,

            "val_srcc": val_srcc

        }

        torch.save(
            best_state,
            best_checkpoint
        )

        print(f"✓ Best Model Saved (SRCC={best_srcc:.4f})")


training_time = (time.time() - start_time) / 60

print("\n============================")
print("Training Complete")
print("============================")
print(f"Training Time : {training_time:.2f} minutes")

results_df = pd.DataFrame(history)

display(results_df)

print("\nHistory:")
print(history_path)

print("\nLatest Checkpoint:")
print(latest_checkpoint)

print("\nBest Checkpoint:")
print(best_checkpoint)


Resuming from Epoch 9

Training Backbone : resnet50_aspp
Epoch 09/20 | Loss=0.2671 | PLCC=0.5090 | SRCC=0.7944
✓ Best Model Saved (SRCC=0.7944)
Epoch 10/20 | Loss=0.2227 | PLCC=0.4927 | SRCC=0.8053
✓ Best Model Saved (SRCC=0.8053)
Epoch 11/20 | Loss=0.2061 | PLCC=0.5176 | SRCC=0.8193
✓ Best Model Saved (SRCC=0.8193)
Epoch 12/20 | Loss=0.1948 | PLCC=0.5012 | SRCC=0.8292
✓ Best Model Saved (SRCC=0.8292)
Epoch 13/20 | Loss=0.1688 | PLCC=0.4919 | SRCC=0.8524
✓ Best Model Saved (SRCC=0.8524)


Load Best Model

In [ ]:
import os

model_path = os.path.join(
    SAVE_DIR,
    f"best_dbiqa_{BACKBONE}.pth"
)

if os.path.exists(model_path):

    checkpoint = torch.load(
        model_path,
        map_location=device,
        weights_only=False
    )

    feature_net.load_state_dict(
        checkpoint["feature_net"]
    )

    model.load_state_dict(
        checkpoint["dbiqa"]
    )

    print(
        f"Loaded Best Model "
        f"(Epoch {checkpoint['epoch']+1})"
    )

else:
    print("No saved model found.")

Final Test

In [ ]:
test_plcc,test_srcc = evaluate(
    test_loader
)

final_results = pd.DataFrame({
    "Backbone":[BACKBONE],
    "PLCC":[test_plcc],
    "SRCC":[test_srcc],
    "Training_Time_Minutes":[round((time.time()-start_time)/60,2)]
})

final_results.to_csv(
    os.path.join(
        SAVE_DIR,
        f"DBIQA_{BACKBONE}_FinalResults.csv"
    ),
    index=False
)

print("Results Saved Successfully")

print("\nFINAL RESULTS")
print("PLCC :",test_plcc)
print("SRCC :",test_srcc)

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/DBIQA_Results/KADID10K"
os.makedirs(SAVE_DIR, exist_ok=True)

results_path = os.path.join(
    SAVE_DIR,
    f"DBIQA_{BACKBONE}_FinalResults.csv"
)

total_time = time.time() - start_time

final_results = pd.DataFrame({
    "Backbone": [BACKBONE],
    "PLCC": [test_plcc],
    "SRCC": [test_srcc],
    "Training_Time_Minutes": [round(total_time/60, 2)]
})
results_path=os.path.join(
    SAVE_DIR,
    f"DBIQA_{BACKBONE}_FinalResults.csv"
)

final_results.to_csv(
    results_path,
    index=False
)

print("Final Results Saved")

final_results.to_csv(results_path, index=False)

ckpt_path = os.path.join(
    SAVE_DIR,
    f"best_dbiqa_{BACKBONE}.pth"
)

Save Results

In [ ]:
final_results = pd.DataFrame({
    "Backbone": [BACKBONE],
    "PLCC": [test_plcc],
    "SRCC": [test_srcc]
})

display(final_results)

final_results.to_csv(
    f"DBIQA_{BACKBONE}_FinalResults.csv",
    index=False
)

In [ ]:
import pandas as pd
import glob

files = glob.glob(
    "/content/drive/MyDrive/DBIQA_Results/KADID10K/*FinalResults.csv"
)

if len(files) == 0:
    print("No result files found.")
else:
    dfs = [pd.read_csv(f) for f in files]

    leaderboard = pd.concat(dfs, ignore_index=True)

    leaderboard = leaderboard.sort_values(
        by="SRCC",
        ascending=False
    ).reset_index(drop=True)

    leaderboard.index += 1
    leaderboard.index.name = "Rank"

    print("\n=== LEADERBOARD ===\n")

    display(leaderboard)
